In [ ]:
from google.colab import files
uploaded=files.upload()


Saving ab_data.csv to ab_data.csv


In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.stats.api as sms
import seaborn as sns
from math import ceil
df=pd.read_csv("ab_data.csv")
df.head()

,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


In [ ]:
# This dataset contains user IDs, session timestamp of accessing the landing page, and information on whether the user belongs to the control or the treatment group,
# converted or not. This dataset contains multiple session data for some users. We will remove users with multiple sessions from our testing.

In [ ]:
# Users with multiple session counts
session_counts = df['user_id'].value_counts(ascending = False)
session_counts.head()
#  If a single user has multiple sessions, their behavior across those sessions is not independent.
# Counting multiple sessions from the same user as separate data points can violate this assumption

,count
user_id,
752737,2
781280,2
767913,2
886060,2
731779,2


In [ ]:
multi_users = session_counts[session_counts > 1].count()
print(multi_users)

3894


In [ ]:
# dropping with multiple session counts
users_to_drop = session_counts[session_counts > 1].index
print(users_to_drop)

Index([752737, 781280, 767913, 886060, 731779, 827432, 677313, 756804, 671874,
       741541,
       ...
       716103, 831390, 633243, 883524, 817288, 835630, 931722, 808992, 668683,
       866251],
      dtype='int64', name='user_id', length=3894)


In [ ]:
df = df[~ df['user_id'].isin(users_to_drop)]
print(df)

        user_id                   timestamp      group landing_page  converted
0        851104  2017-01-21 22:11:48.556739    control     old_page          0
1        804228  2017-01-12 08:01:45.159739    control     old_page          0
2        661590  2017-01-11 16:55:06.154213  treatment     new_page          0
3        853541  2017-01-08 18:28:03.143765  treatment     new_page          0
4        864975  2017-01-21 01:52:26.210827    control     old_page          1
...         ...                         ...        ...          ...        ...
294473   751197  2017-01-03 22:28:38.630509    control     old_page          0
294474   945152  2017-01-12 00:51:57.078372    control     old_page          0
294475   734608  2017-01-22 11:45:03.439544    control     old_page          0
294476   697314  2017-01-15 01:20:28.957438    control     old_page          0
294477   715931  2017-01-16 12:40:24.467417  treatment     new_page          0

[286690 rows x 5 columns]


In [ ]:
# Next, we will randomly sample 5000 users' data from each group and perform the statistical significance test.

In [ ]:
# Sample control and treatment group
control_sample = df[df['group'] == 'control'].sample(n = 5000, random_state = 12)
treatment_sample = df[df['group'] == 'treatment'].sample(n = 5000, random_state = 12)
ab_test = pd.concat([control_sample, treatment_sample], axis=0)
ab_test.reset_index(drop=True, inplace=True)
ab_test

,user_id,timestamp,group,landing_page,converted
0,722876,2017-01-03 16:24:40.954412,control,old_page,0
1,630561,2017-01-16 12:06:13.597159,control,old_page,0
2,704591,2017-01-21 06:42:13.850395,control,old_page,0
3,651659,2017-01-23 00:08:59.556011,control,old_page,0
4,733154,2017-01-04 11:05:36.544602,control,old_page,0
...,...,...,...,...,...
9995,736983,2017-01-09 22:31:51.401942,treatment,new_page,0
9996,778410,2017-01-17 07:26:12.569399,treatment,new_page,0
9997,865923,2017-01-14 22:11:18.607758,treatment,new_page,0
9998,842876,2017-01-07 07:05:42.651190,treatment,new_page,0


In [ ]:
# Define functions for standard deviation and standard error
std_dev = lambda x : np.std(x, ddof = 0)
std_error = lambda x : stats.sem(x, ddof = 0)
conversion_rate = ab_test.groupby('group')['converted'].agg([np.mean, std_dev, std_error])
conversion_rate.columns = ['conversion_rate', 'std_deviation', 'std_error']
conversion_rate

/tmp/ipykernel_977/1398751122.py:4: FutureWarning: The provided callable <function mean at 0x7c4f81d21620> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  conversion_rate = ab_test.groupby('group')['converted'].agg([np.mean, std_dev, std_error])


,conversion_rate,std_deviation,std_error
group,,,
control,0.1144,0.318296,0.004501
treatment,0.1218,0.327055,0.004625


In [ ]:

from statsmodels.stats.proportion import proportions_ztest, proportion_confint

control_results = ab_test[ab_test['group'] == 'control']['converted']
treatment_results = ab_test[ab_test['group'] == 'treatment']['converted']


num_control = control_results.count()
num_control

np.int64(5000)

In [ ]:
num_treatment = treatment_results.count()
num_treatment

np.int64(5000)

In [ ]:
successes = [control_results.sum(), treatment_results.sum()]
successes

[np.int64(572), np.int64(609)]

In [ ]:
nobs = [num_control, num_treatment]
nobs

[np.int64(5000), np.int64(5000)]

In [ ]:
z_stat, pval = proportions_ztest(successes, nobs = nobs)
(lower_con, lower_treat), (upper_con, upper_treat) = proportion_confint(successes, nobs=nobs, alpha=0.05)
lower_con, lower_treat,upper_con, upper_treat,z_stat,pval

(np.float64(0.1055774342215452),
 np.float64(0.11273467352153901),
 np.float64(0.1232225657784548),
 np.float64(0.130865326478461),
 np.float64(-1.1464816392633446),
 np.float64(0.25159592004286013))

In [ ]:
print(f'Z Statistic - {z_stat:.2f}')
print(f'P-Value - {pval:.3f}')
print(f'CI 95% for control group - [{lower_con:.3f}, {upper_con:.3f}]')
print(f'CI 95% for treatment group - [{lower_treat:.3f}, {upper_treat:.3f}]')

Z Statistic - -1.15
P-Value - 0.252
CI 95% for control group - [0.106, 0.123]
CI 95% for treatment group - [0.113, 0.131]


In [ ]:
##  https://www.scaler.com/topics/data-science/a-b-testing-in-python/  --Check for A/B testing

In [ ]:
# As you can see, p-value for the A/B testing results is 0.252. If we set the significance level at 0.05, then we can’t reject the NULL hypothesis at this p-value.
#  It means that observed results have occurred due to random chance and are not statistically significant to say that the landing page of the treatment group works better.

# Conclusion
# A/B testing compares two versions of a product, website, app, or marketing campaign to determine which one performs better.
# It is useful in a situation when you want to test the effectiveness of incremental changes in a product, website, or app.
# Post conducting A/B testing, we need to prove that the results are statistically significant. It means that the observed results are not due to some random chance.